In [12]:
import sys
sys.path.insert(0, "..")

import pystac
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import pickle
from datetime import datetime
import rioxarray as rxr
from scipy.stats import randint, uniform

# --- RF ---
from xgboost import XGBRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from helpers.shared import time_match

In [13]:
from config import *

# -- Catalog --------------------------------------------------------------------------------------------
CATALOG                 = load_catalog()
INPUT_ASSET             = "train-stack"

# --- Bands ---
# DROP_BANDS = {
#                 "wv2-imagery" : ["REdge", "NIR1", "NIR2"],
#                 "sd8-imagery" : ["G2", "REdge", "NIR1"]
# }

# REASSIGN_BANDS = {
#                 "G1" : "G"
#                 }

# --- Output Options ---
CACHE_DIR = Path("../out/rf/")

# --- Algorithm Parameters ---

SENSOR_BIN = {
                "wv2-imagery": 0,
                "sd8-imagery": 1
                }

METHOD = XGBRegressor

PARAMS = {
    RandomForestRegressor: CACHE_DIR / "hyperparam_search_RandomForestRegressor_20260428-123806.json",
    XGBRegressor: CACHE_DIR / "hyperparam_search_XGBRegressor_20260428-120837.json",
}

PARAM_DIST = {
    RandomForestRegressor: dict(
        n_estimators      = randint(100, 600),
        max_features      = ["sqrt", "log2", 0.3, 0.5],
        min_samples_leaf  = randint(1, 20),
        max_depth         = [None, 10, 20, 30],
    ),
    XGBRegressor: dict(
        n_estimators      = randint(200, 800),
        max_depth         = [3, 4, 5, 6, 8],
        learning_rate     = uniform(0.01, 0.19),
        subsample         = uniform(0.6, 0.4),
        colsample_bytree  = uniform(0.5, 0.5),
        min_child_weight  = randint(1, 20),
        reg_alpha         = uniform(0, 1),
        reg_lambda        = uniform(0.5, 2),
    ),
}


In [14]:
stack_col = CATALOG.get_child("stacks")
train_stack = stack_col.assets[INPUT_ASSET]
sampled_df = pd.read_parquet(train_stack.href)

In [24]:
print(list(sampled_df.columns))
print(sampled_df.shape)

['x', 'y', 'elev', 'sg_percov', 'block_x', 'block_y', 'block_id', 'B', 'G', 'Y', 'R', 'sg_species', 'sensor', 'tide_h', 'a_B', 'a_G', 'a_R', 'Kd_G', 'zSD', 'log_B', 'log_G', 'log_Y', 'log_R', 'nd_B_G', 'nd_B_Y', 'nd_B_R', 'nd_G_Y', 'nd_G_R', 'nd_Y_R']
(540346, 29)


In [15]:
# --- Features ---
DROP = {"x", "y", "block_id", "block_x", "block_y", "elev", "sensor"}
FEATURE_COLS = [c for c in sampled_df.columns if c not in DROP]
RESPONSE = "elev"
ROUTER = "sensor"
GROUP = "block_id"

In [16]:
sampled_df.groupby("sensor").size()

sensor
0    261428
1    278918
dtype: int64

In [17]:
X = sampled_df[FEATURE_COLS].values
y = sampled_df[RESPONSE].values
router = sampled_df[ROUTER].values
blocks = sampled_df[GROUP].values

In [18]:
# --- Stage 1 domain pruning ---

DROP_FEATS = (
    {'B', 'G', 'Y', 'R'} # drop in favour of log bands
    # {f for f in FEATURE_COLS if f.startswith('log_')} # redundant with normal bands
    | {'a_B'} # collinear with Kd_G; Kd_G more interpretable
    # | {'tide_h'} # 1:1 relationship with sensor; i.e one tide state per sensor
    # | {f for f in FEATURE_COLS if f.startswith('nd_') and f != 'nd_G_R'} # keep only normalised difference
)

KEPT_FEATURES = [f for f in FEATURE_COLS if f not in DROP_FEATS]

X = sampled_df[KEPT_FEATURES].values

In [19]:
import json
best_params = json.loads(PARAMS[METHOD].read_text())["best_params"]

NOTE = None

In [20]:
# # --- Shared fold assignment ---
# all_blocks = np.array(sorted(sampled_df[GROUP].unique()))
# _dummy     = np.zeros(len(all_blocks))
# fold_map   = {}
# for fold, (_, val_idx) in enumerate(GroupKFold(n_splits=N_FOLDS).split(_dummy, groups=all_blocks)):
#     for b in all_blocks[val_idx]:
#         fold_map[b] = fold

# results     = []
# importances = []

# for sensor_binary in tqdm(sorted(sampled_df[ROUTER].unique()), desc="Expert RF:"):
#     sensor = next(k for k, v in SENSOR_BIN.items() if v == sensor_binary)
#     tqdm.write(f"=================== {sensor} ==================")

#     s = router == sensor_binary
#     X_s, y_s, blocks_s = X[s], y[s], blocks[s]
#     folds_s = np.array([fold_map[b] for b in blocks_s])

#     for fold in tqdm(range(N_FOLDS), total=N_FOLDS, leave=False):
#         tr  = np.where(folds_s != fold)[0]
#         val = np.where(folds_s == fold)[0]

#         m = METHOD(**best_params)
#         m.fit(X_s[tr], y_s[tr])
#         preds = m.predict(X_s[val])

#         val_blocks = np.unique(blocks_s[val])
#         val_depth  = y_s[val]
#         tqdm.write(f"Fold {fold} | blocks={val_blocks} | n={len(val)} | depth [{val_depth.min():.2f}, {val_depth.max():.2f}] | mean={val_depth.mean():.2f}")

#         results.append({
#             "sensor"     : sensor,
#             "sensor_bin" : sensor_binary,
#             "fold"       : fold,
#             "mae"        : mean_absolute_error(y_s[val], preds),
#             "r2"         : r2_score(y_s[val], preds),
#             "rmse"       : np.sqrt(mean_squared_error(y_s[val], preds)),
#             "n_train"    : len(tr),
#             "n_val"      : len(val),
#             "modeller"   : METHOD.__name__,
#             "params"     : best_params,
#             "predictors" : KEPT_FEATURES,
#             "note"       : NOTE,
#         })

#         importances.append({
#             "sensor" : sensor,
#             "fold"   : fold,
#             **dict(zip(KEPT_FEATURES, m.feature_importances_))
#         })

#         tqdm.write(f"Fold {fold} | MAE = {results[-1]['mae']:.3f} | RMSE = {results[-1]['rmse']:.3f} | R2 = {results[-1]['r2']:.3f}")

#     sensor_rows = results[-N_FOLDS:]
#     sensor_df   = pd.DataFrame(sensor_rows)
#     tqdm.write(f"Mean | MAE={sensor_df['mae'].mean():.3f} | RMSE={sensor_df['rmse'].mean():.3f} | R2={sensor_df['r2'].mean():.3f}")


In [21]:
# import matplotlib.pyplot as plt
# import matplotlib.patches as mpatches

# fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharex=True, sharey=True)

# fold_colors = plt.cm.tab10.colors

# for ax, sensor_binary in zip(axes, sorted(sampled_df[ROUTER].unique())):
#     sensor = next(k for k, v in SENSOR_BIN.items() if v == sensor_binary)
#     sdf = sampled_df[sampled_df[ROUTER] == sensor_binary].copy()
#     sdf["fold"] = sdf[GROUP].map(fold_map)

#     for fold in range(N_FOLDS):
#         fdf = sdf[sdf["fold"] == fold]
#         ax.scatter(fdf["x"], fdf["y"], c=[fold_colors[fold]], s=0.1, alpha=0.3, rasterized=True)

#         # label block centroids
#         for block_id, bdf in fdf.groupby(GROUP):
#             ax.text(bdf["x"].mean(), bdf["y"].mean(), str(block_id),
#                     fontsize=8, ha="center", va="center", fontweight="bold")

#     ax.set_title(sensor)
#     ax.set_aspect("equal")
#     ax.set_xlabel("x (m)")
#     ax.set_ylabel("y (m)")

# patches = [mpatches.Patch(color=fold_colors[f], label=f"Fold {f}") for f in range(N_FOLDS)]
# axes[1].legend(handles=patches, loc="upper right", fontsize=8)

# plt.suptitle("Spatial block assignments by fold and sensor")
# plt.tight_layout()
# plt.show()


In [22]:
results = []
importances = []

for sensor_binary in tqdm(sorted(sampled_df[ROUTER].unique()), desc = "Expert RF:"):
    sensor = next(k for k, v in SENSOR_BIN.items() if v == sensor_binary)
    tqdm.write(f"=================== {sensor} ==================")

    s = router == sensor_binary

    X_s, y_s, blocks_s = X[s], y[s], blocks[s]

    kfold   = GroupKFold(n_splits=N_FOLDS)

    for fold, (tr, val) in enumerate(tqdm(kfold.split(X_s, y_s, blocks_s), total=N_FOLDS, leave = False)):
        m = METHOD(**best_params)
        m.fit(X_s[tr], y_s[tr])
        preds = m.predict(X_s[val])

        val_blocks = np.unique(blocks_s[val])
        val_depth  = y_s[val]
        tqdm.write(f"Fold {fold} | blocks={val_blocks} | n={len(val)} | depth [{val_depth.min():.2f}, {val_depth.max():.2f}] | mean={val_depth.mean():.2f}")

        results.append({
            "sensor"     : sensor,
            "sensor_bin" : sensor_binary,
            "fold"       : fold,
            "mae"        : mean_absolute_error(y_s[val], preds),
            "r2"         : r2_score(y_s[val], preds),
            "rmse"       : np.sqrt(mean_squared_error(y_s[val], preds)),
            "n_train"    : len(tr),
            "n_val"      : len(val),
            "modeller"   : METHOD.__name__,
            "params"     : best_params,
            "predictors" : KEPT_FEATURES,
            "note"       : NOTE,
        })

        importances.append({
            "sensor" : sensor,
            "fold"   : fold,
            **dict(zip(KEPT_FEATURES, m.feature_importances_))
        })

        tqdm.write(f"Fold {fold} | MAE = {results[-1]['mae']:.3f} | RMSE = {results[-1]['rmse']:.3f} | R2 = {results[-1]['r2']:.3f}")

    sensor_rows = results[-N_FOLDS:]
    sensor_df = pd.DataFrame(sensor_rows)
    tqdm.write(f"Mean | MAE={sensor_df['mae'].mean():.3f} | RMSE={sensor_df['rmse'].mean():.3f} | R2={sensor_df['r2'].mean():.3f}")

cv_df = pd.DataFrame(results)
fname = datetime.now().strftime("kfold-experts-%Y%m%d-%H%M%S.pkl")

with open(CACHE_DIR / fname, "wb") as f:
    pickle.dump(cv_df, f)


Expert RF::   0%|          | 0/2 [00:00<?, ?it/s]

=================== wv2-imagery ==================


  0%|          | 0/5 [00:00<?, ?it/s]

Fold 0 | blocks=[10] | n=50855 | depth [-3.00, -0.02] | mean=-1.75
Fold 0 | MAE = 0.130 | RMSE = 0.249 | R2 = 0.871
Fold 1 | blocks=[3 8] | n=35996 | depth [-3.00, -0.02] | mean=-2.55
Fold 1 | MAE = 0.129 | RMSE = 0.179 | R2 = 0.859
Fold 2 | blocks=[0 5] | n=48847 | depth [-3.00, -0.02] | mean=-1.79
Fold 2 | MAE = 0.132 | RMSE = 0.180 | R2 = 0.968
Fold 3 | blocks=[4 6] | n=62570 | depth [-3.00, -0.05] | mean=-1.93
Fold 3 | MAE = 0.101 | RMSE = 0.152 | R2 = 0.953
Fold 4 | blocks=[1 7] | n=63160 | depth [-3.00, -0.02] | mean=-1.61
Fold 4 | MAE = 0.112 | RMSE = 0.162 | R2 = 0.948
Mean | MAE=0.121 | RMSE=0.184 | R2=0.920
=================== sd8-imagery ==================


  0%|          | 0/5 [00:00<?, ?it/s]

Fold 0 | blocks=[1] | n=37563 | depth [-3.00, -0.10] | mean=-1.44
Fold 0 | MAE = 0.315 | RMSE = 0.444 | R2 = 0.650
Fold 1 | blocks=[ 6 10] | n=53919 | depth [-3.00, -0.15] | mean=-1.58
Fold 1 | MAE = 0.193 | RMSE = 0.292 | R2 = 0.881
Fold 2 | blocks=[7 8] | n=63292 | depth [-3.00, -0.14] | mean=-1.45
Fold 2 | MAE = 0.258 | RMSE = 0.358 | R2 = 0.753
Fold 3 | blocks=[4 5] | n=62234 | depth [-3.00, -0.12] | mean=-1.51
Fold 3 | MAE = 0.249 | RMSE = 0.350 | R2 = 0.797
Fold 4 | blocks=[0 3] | n=61910 | depth [-3.00, -0.17] | mean=-2.16
Fold 4 | MAE = 0.276 | RMSE = 0.405 | R2 = 0.465
Mean | MAE=0.258 | RMSE=0.370 | R2=0.709


In [23]:
imp_df = pd.DataFrame(importances)
imp_df.groupby("sensor")[KEPT_FEATURES].mean().T.sort_values("wv2-imagery", ascending=False)

sensor,sd8-imagery,wv2-imagery
nd_G_R,0.033155,0.466998
nd_B_R,0.039814,0.344989
a_G,0.022385,0.065026
nd_B_Y,0.022325,0.049510
log_R,0.302908,0.032230
nd_G_Y,0.032272,0.008777
nd_Y_R,0.005641,0.005115
log_B,0.007880,0.004887
sg_percov,0.025588,0.004639
sg_species,0.024047,0.003633
